# Detecção de Fadiga em Motoristas
## Análise Exploratória e Visualização dos Resultados

Este notebook complementa o pipeline de detecção de fadiga.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import cv2
from src.config import *
from src.landmarks import calculate_ear, get_landmarks, get_face_mesh_instance

## 1. Visualização dos Landmarks Faciais (MediaPipe Face Mesh)

In [ ]:
cap = cv2.VideoCapture(0)
ret, frame = cap.read()
cap.release()

if ret:
    import mediapipe as mp
    mp_drawing = mp.solutions.drawing_utils
    mp_face_mesh = mp.solutions.face_mesh

    with mp_face_mesh.FaceMesh(max_num_faces=1, refine_landmarks=True) as fm:
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = fm.process(rgb)
        if results.multi_face_landmarks:
            annotated = frame.copy()
            mp_drawing.draw_landmarks(
                annotated,
                results.multi_face_landmarks[0],
                mp_face_mesh.FACEMESH_TESSELATION,
                landmark_drawing_spec=mp_drawing.DrawingSpec(color=(0,255,0), thickness=1, circle_radius=1),
            )
            plt.figure(figsize=(10,8))
            plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
            plt.title('MediaPipe Face Mesh - 468 Landmarks')
            plt.axis('off')
            plt.show()

## 2. Cálculo do EAR - Demonstração

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.text(0.5, 0.5,
        r'$EAR = \frac{||p_2 - p_6|| + ||p_3 - p_5||}{2 \cdot ||p_1 - p_4||}$',
        fontsize=24, ha='center', va='center', transform=ax.transAxes)
ax.set_title('Eye Aspect Ratio (EAR)', fontsize=16)
ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Resultados em Tempo Real

In [ ]:
from PIL import Image

result_files = {
    'EAR/MAR em Tempo Real': 'ear_mar_realtime.png',
}

for title, filename in result_files.items():
    filepath = os.path.join(RESULTS_DIR, filename)
    if os.path.exists(filepath):
        img = Image.open(filepath)
        plt.figure(figsize=(12, 6))
        plt.imshow(img)
        plt.title(title, fontsize=14)
        plt.axis('off')
        plt.show()
    else:
        print(f'{title}: Arquivo não encontrado ({filepath})')

## 4. Métricas de Inferência

In [ ]:
metrics_file = os.path.join(RESULTS_DIR, 'inference_metrics.txt')
if os.path.exists(metrics_file):
    with open(metrics_file) as f:
        print(f.read())
else:
    print('Execute a detecção primeiro: python main.py detect')